[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShintaroMinami/notebook_proteina_complexa/blob/main/proteina_complexa.ipynb)

# **Proteina Complexa — Hands-On Notebook**

このノートブックでは、NVIDIAが開発したタンパク質デザインモデル **Proteina Complexa** を
Google Colab上で実行します。

> **必要なもの:** Googleアカウントのみ。ローカル環境の構築は不要です。

In [ ]:
#@title 初期設定
#@markdown このセルを最初に実行してください。
#@markdown
#@markdown ---
#@markdown ### Google Drive の使用
#@markdown - **ON**: 初回はダウンロード後 Drive に保存。2回目以降は Drive から復元（高速）
#@markdown - **OFF**: 毎回インターネットからダウンロード
use_google_drive = False #@param {type:"boolean"}

import subprocess
from IPython.display import display, HTML

def run(cmd, title=""):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    lines = output.split("\n")
    icon = "\u2705" if result.returncode == 0 else "\u274c"
    summary = f"{icon} {title} ({len(lines)} lines)" if title else f"{icon} Output ({len(lines)} lines)"
    html = (
        f'<details><summary style="cursor:pointer;font-weight:bold;padding:4px 0">{summary}</summary>'
        f'<pre style="max-height:300px;overflow-y:auto;background:#f5f5f5;padding:8px;'
        f'font-size:11px;border-radius:4px;margin-top:4px">{output}</pre>'
        f'</details>'
    )
    display(HTML(html))
    return result.returncode

if use_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CACHE = "/content/drive/MyDrive/.proteina_complexa_cache"
    import os
    os.makedirs(DRIVE_CACHE, exist_ok=True)
    print(f"Google Drive cache: {DRIVE_CACHE}")
else:
    DRIVE_CACHE = None
    print("Google Drive: OFF（毎回ダウンロードします）")

print("Ready.")


---
## **1. 環境構築**

はじめに、必要なソフトウェアをインストールします。
以下のセルを **上から順に** 実行してください（各セルで Shift+Enter）。

In [ ]:
#@title 1.1 GPU の確認
#@markdown GPUが割り当てられているか確認します。わり当てられていない場合は、ランタイムのタイプを変更してください。
#@markdown
#@markdown ---

import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"GPU: {gpu_info}")
else:
    print("GPU が見つかりません。ランタイムのタイプを変更してください。")

print(f"Python: {sys.version}")

In [ ]:
#@title 1.2 ソフトウェアのインストール
#@markdown 必要なソフトウェアを順次インストールします。実行には10〜15分程度かかります。
#@markdown
#@markdown ---

import os

REPO_DIR = "/content/Proteina-Complexa"

# --- uv + repository ---
run("pip install -q uv", "uv install")

if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    run("git clone --depth 1 https://github.com/NVIDIA-Digital-Bio/Proteina-Complexa.git /content/Proteina-Complexa", "git clone")

os.chdir(REPO_DIR)
print(f"リポジトリ: {os.getcwd()}")

# --- PyTorch ---
run(
    "uv pip install --system torch==2.7.0+cu126 torchvision torchaudio"
    " --index-url https://download.pytorch.org/whl/cu126",
    "PyTorch + CUDA 12.6"
)

# --- Proteina Complexa ---
run("uv pip install --system --index-strategy unsafe-best-match -e .", "Proteina Complexa")
run(
    "uv pip install --system torch_geometric torch_scatter torch_sparse torch_cluster"
    " -f https://data.pyg.org/whl/torch-2.7.0+cu126.html",
    "PyTorch Geometric"
)
run("uv pip install --system graphein==1.7.7 --no-deps", "Graphein")
run("uv pip install --system \"atomworks[ml,openbabel,dev]\" 2>/dev/null || true", "Atomworks")
run("uv pip install --system py3Dmol", "py3Dmol")
run("uv pip install --system biotite==1.6.0", "biotite")

# --- ColabDesign (AF2) ---
run("uv pip install --system colabdesign==1.1.1 alphafold-colabfold==2.3.7", "ColabDesign")
run(f"uv pip install --system -e {REPO_DIR}/community_models/colabdesign", "ColabDesign (local)")
run(
    "uv pip install --system jaxlib==0.4.29+cuda12.cudnn91"
    " -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    "jaxlib (CUDA)"
)
run(
    "uv pip install --system 'jax[cuda12]==0.4.29'"
    " -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    "JAX (CUDA)"
)
run("uv pip install --system flax==0.9.0 --no-deps", "Flax")

# --- Boltz-2 ---
run("uv pip install --system --index-strategy unsafe-best-match \"boltz[cuda]\"", "Boltz-2")
run(
    "uv pip install --system --reinstall torch==2.7.0+cu126 torchvision torchaudio"
    " --index-url https://download.pytorch.org/whl/cu126",
    "PyTorch reinstall"
)
run("uv pip install --system \"numpy<2.1\"", "NumPy pin")

import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print("Done.")

In [ ]:
#@title 1.3 モデルファイルをダウンロード
#@markdown ---
#@markdown | モデル | 用途 | サイズ |
#@markdown |---|---|---|
#@markdown | Protein Target | タンパク質に結合するバインダーの設計 | 約 6.6 GB |
#@markdown | Ligand Target | 低分子リガンドに結合するバインダーの設計 | 約 5.9 GB |
#@markdown | AME | 機能モチーフのスキャフォールディング | 約 5.9 GB |
#@markdown | AF2 | 自己一貫性検証（ColabDesign） | 約 5.4 GB |
#@markdown | Boltz-1 | 構造予測・検証 | 約 1.5 GB |
#@markdown ---
download_protein_target = True #@param {type:"boolean"}
download_ligand_target = False #@param {type:"boolean"}
download_ame = False #@param {type:"boolean"}
download_af2 = True #@param {type:"boolean"}
download_boltz = True #@param {type:"boolean"}

import os, shutil
from pathlib import Path

# --- Proteina Complexa ---
if not os.path.exists(".env"):
    run("complexa init", "Proteina Complexa init")
else:
    print("Proteina Complexa: .env already exists, skipping init.")

os.environ["COMPLEXA_INIT"] = "1"
os.environ["LOCAL_CODE_PATH"] = "/content/Proteina-Complexa"
os.environ["DATA_PATH"] = "/content/Proteina-Complexa/assets"

complexa_models = []
if download_protein_target:
    complexa_models.append("--complexa")
if download_ligand_target:
    complexa_models.append("--complexa-ligand")
if download_ame:
    complexa_models.append("--complexa-ame")

if complexa_models:
    dl_flags = " ".join(complexa_models)
    complexa_cache = Path(DRIVE_CACHE) / "complexa" if DRIVE_CACHE else None
    complexa_local = Path("checkpoints")

    if complexa_cache and complexa_cache.exists() and any(complexa_cache.iterdir()):
        print("Proteina Complexa weights: restoring from Google Drive...")
        if complexa_local.exists():
            shutil.rmtree(complexa_local)
        shutil.copytree(complexa_cache, complexa_local)
        print("Proteina Complexa weights: restored.")
    else:
        run(f"complexa download {dl_flags}", "Proteina Complexa weights")
        if complexa_cache and complexa_local.exists():
            print("Saving Proteina Complexa weights to Google Drive...")
            if complexa_cache.exists():
                shutil.rmtree(complexa_cache)
            shutil.copytree(complexa_local, complexa_cache)
            print("Saved.")

# --- AF2 (AlphaFold2 multimer) ---
AF2_PARAMS_DIR = Path("/content/af2_params")

if download_af2:
    af2_cache = Path(DRIVE_CACHE) / "af2_params" if DRIVE_CACHE else None

    if af2_cache and af2_cache.exists() and any(af2_cache.iterdir()):
        print("AF2 weights: linking from Google Drive...")
        if AF2_PARAMS_DIR.exists() or AF2_PARAMS_DIR.is_symlink():
            AF2_PARAMS_DIR.unlink() if AF2_PARAMS_DIR.is_symlink() else shutil.rmtree(AF2_PARAMS_DIR)
        AF2_PARAMS_DIR.symlink_to(af2_cache)
        print("AF2 weights: ready (symlink).")
    elif AF2_PARAMS_DIR.exists() and any(AF2_PARAMS_DIR.glob("*.npz")):
        print("AF2 weights: already downloaded.")
    else:
        AF2_PARAMS_DIR.mkdir(parents=True, exist_ok=True)
        run("wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar -O /tmp/af2_params.tar", "AF2 weights download")
        run(f"tar -xf /tmp/af2_params.tar -C {AF2_PARAMS_DIR}", "AF2 weights extract")
        run("rm -f /tmp/af2_params.tar")
        if af2_cache:
            print("Saving AF2 weights to Google Drive...")
            if af2_cache.exists():
                shutil.rmtree(af2_cache)
            shutil.copytree(AF2_PARAMS_DIR, af2_cache)
            shutil.rmtree(AF2_PARAMS_DIR)
            AF2_PARAMS_DIR.symlink_to(af2_cache)
            print("Saved.")

# --- Boltz-1 ---
if download_boltz:
    boltz_cache = Path(DRIVE_CACHE) / "boltz" if DRIVE_CACHE else None
    boltz_local = Path("/root/.boltz")

    if boltz_cache and boltz_cache.exists() and any(boltz_cache.iterdir()):
        print("Boltz-1 weights: linking from Google Drive...")
        if boltz_local.exists() or boltz_local.is_symlink():
            boltz_local.unlink() if boltz_local.is_symlink() else shutil.rmtree(boltz_local)
        boltz_local.symlink_to(boltz_cache)
        print("Boltz-1 weights: ready (symlink).")
    else:
        run("python -c \"from pathlib import Path; p = Path('/root/.boltz'); p.mkdir(parents=True, exist_ok=True); from boltz.main import download_boltz1; download_boltz1(p)\"", "Boltz-1 download")
        if boltz_cache and boltz_local.exists():
            print("Saving Boltz-1 weights to Google Drive...")
            if boltz_cache.exists():
                shutil.rmtree(boltz_cache)
            shutil.copytree(boltz_local, boltz_cache)
            shutil.rmtree(boltz_local)
            boltz_local.symlink_to(boltz_cache)
            print("Saved.")

print("All selected models ready.")

In [ ]:
#@title 1.4 インストールの確認
#@markdown ソフトウェアとモデルファイルのインストールが正しく行われたか確認します。
#@markdown
#@markdown ---

from pathlib import Path
import torch

all_ok = True

print("Software")
print("─" * 40)
print(f"  PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

import shutil
if shutil.which("complexa"):
    print("  Proteina Complexa CLI: OK")
else:
    print("  Proteina Complexa CLI: NOT FOUND")
    all_ok = False

try:
    import colabdesign
    print(f"  ColabDesign: OK ({colabdesign.__version__})")
except ImportError:
    print("  ColabDesign: NOT FOUND")
    all_ok = False

try:
    import jax
    print(f"  JAX: OK ({jax.__version__}, {jax.devices()[0].platform})")
except ImportError:
    print("  JAX: NOT FOUND")
    all_ok = False

print()
print("Model weights")
print("─" * 40)

ckpt_dir = Path("/content/Proteina-Complexa/checkpoints")
weight_checks = {
    "Protein Target": ("complexa.ckpt", download_protein_target),
    "Ligand Target": ("complexa_ligand.ckpt", download_ligand_target),
    "AME": ("complexa_ame.ckpt", download_ame),
}

for name, (filename, requested) in weight_checks.items():
    if not requested:
        continue
    path = ckpt_dir / filename
    if path.exists():
        size_gb = path.stat().st_size / (1024**3)
        print(f"  {name}: OK ({size_gb:.1f} GB)")
    else:
        print(f"  {name}: NOT FOUND ({path})")
        all_ok = False

if download_af2:
    if AF2_PARAMS_DIR.exists() and any(AF2_PARAMS_DIR.glob("*multimer_v3*")):
        n_files = len(list(AF2_PARAMS_DIR.glob("*.npz")))
        print(f"  AF2: OK ({n_files} param files)")
    else:
        print(f"  AF2: NOT FOUND ({AF2_PARAMS_DIR})")
        all_ok = False

if download_boltz:
    boltz_dir = Path("/root/.boltz")
    if boltz_dir.exists() and any(boltz_dir.iterdir()):
        total = sum(f.stat().st_size for f in boltz_dir.rglob("*") if f.is_file())
        print(f"  Boltz-1: OK ({total / (1024**3):.1f} GB)")
    else:
        print(f"  Boltz-1: NOT FOUND ({boltz_dir})")
        all_ok = False

print()
if all_ok:
    print("All checks passed!")
else:
    print("Some checks failed. Please re-run the corresponding cells above.")

---
## **2 プロジェクトの準備**

In [ ]:
#@title 2.1 プロジェクトの作成
#@markdown プロジェクト名を入力してください。
#@markdown
#@markdown ---
project_name = "PDL1_binder_design" #@param {type:"string"}

from pathlib import Path

PROJECT_DIR = Path(f"/content/projects/{project_name}")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "targets").mkdir(exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)
(PROJECT_DIR / "analysis").mkdir(exist_ok=True)

print(f"Project: {project_name}")
print(f"Directory: {PROJECT_DIR}")
print(f"")
print(f"  targets/   ... ターゲットPDBファイル")
print(f"  results/   ... デザイン結果")
print(f"  analysis/  ... 解析結果")

In [ ]:
#@title 2.2 ターゲットの指定
#@markdown - **PDB ID**: RCSB PDBからダウンロードします（例: `3BIK`, `7BQD`）
#@markdown - **Upload**: 手元のPDBファイルをアップロードします
#@markdown
#@markdown ---
target_source = "PDB ID" #@param ["PDB ID", "Upload"]
pdb_id = "3BIK" #@param {type:"string"}

from pathlib import Path
from Bio.PDB import PDBParser
import py3Dmol
import warnings
warnings.filterwarnings("ignore")

target_dir = PROJECT_DIR / "targets"

if target_source == "PDB ID":
    pdb_id = pdb_id.strip().upper()
    target_pdb = target_dir / f"{pdb_id}.pdb"
    if target_pdb.exists():
        print(f"Already downloaded: {target_pdb}")
    else:
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        run(f"wget -q -O {target_pdb} {url}", f"Download {pdb_id}")
        if target_pdb.exists() and target_pdb.stat().st_size > 0:
            print(f"Downloaded: {target_pdb}")
        else:
            print(f"Error: PDB ID '{pdb_id}' が見つかりませんでした。IDを確認してください。")
else:
    from google.colab import files
    print("PDBファイルを選択してください:")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        target_pdb = target_dir / filename
        target_pdb.write_bytes(content)
        pdb_id = target_pdb.stem.upper()
        print(f"Uploaded: {target_pdb}")

# Chain info
parser = PDBParser(QUIET=True)
structure = parser.get_structure("target", str(target_pdb))

print(f"\nFile: {target_pdb.name}")
print("─" * 50)
for model in structure:
    for chain in model:
        residues = [r for r in chain.get_residues() if r.id[0] == " "]
        if not residues:
            continue
        first = residues[0].id[1]
        last = residues[-1].id[1]
        print(f"  Chain {chain.id}: residues {first}-{last} ({len(residues)} residues)")
    break
print("─" * 50)

# 3D viewer
with open(target_pdb) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

In [ ]:
#@title 2.3 設計パラメータの設定
#@markdown 設計に使用するターゲットのチェーンと残基範囲、ホットスポット残基、生成するバインダーの長さを指定します。
#@markdown
#@markdown ---
#@markdown ### ターゲット設定
#@markdown - **target_chain_residues**: ターゲットのチェーンと残基範囲（例: `A1-115`）
#@markdown - **hotspot_residues**: バインダーが結合してほしい残基（カンマ区切り、例: `A56,A113,A121`）
#@markdown ### バインダー設定
#@markdown - **binder_length_min / max**: 生成するバインダーの長さの範囲
#@markdown ---
target_chain_residues = "A18-133" #@param {type:"string"}
hotspot_residues = "A56,A113,A115,A121,A123" #@param {type:"string"}
binder_length_min = 60 #@param {type:"integer"}
binder_length_max = 80 #@param {type:"integer"}

hotspot_list = [r.strip() for r in hotspot_residues.split(",") if r.strip()]

target_config = {
    "source": "user_targets",
    "target_filename": target_pdb.stem,
    "target_path": str(target_pdb),
    "target_input": target_chain_residues,
    "hotspot_residues": hotspot_list,
    "binder_length": [binder_length_min, binder_length_max],
    "pdb_id": pdb_id.lower(),
}

print("Target configuration:")
for k, v in target_config.items():
    print(f"  {k}: {v}")
TASK_NAME = f"user_{pdb_id}"
print(f"\nTask name: {TASK_NAME}")

# --- Parse target_chain_residues ---
import re
match = re.match(r"([A-Z])(\d+)-(\d+)", target_chain_residues)
target_chain = match.group(1)
target_resi_start = int(match.group(2))
target_resi_end = int(match.group(3))

# --- Parse hotspot residues ---
hotspot_resis = []
for h in hotspot_list:
    m = re.match(r"([A-Z])(\d+)", h)
    if m:
        hotspot_resis.append(int(m.group(2)))

# --- 3D Viewer ---
import py3Dmol

with open(target_pdb) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, "pdb")

# Non-target region: thin, transparent
view.setStyle(
    {"cartoon": {"color": "lightgray", "opacity": 0.3}}
)

# Target region: colored
view.setStyle(
    {"chain": target_chain, "resi": [f"{target_resi_start}-{target_resi_end}"]},
    {"cartoon": {"color": "skyblue"}}
)

# Hotspot residues: red sticks + cartoon
if hotspot_resis:
    view.setStyle(
        {"chain": target_chain, "resi": hotspot_resis},
        {"cartoon": {"color": "red"}, "stick": {"color": "red"}}
    )

view.zoomTo({"chain": target_chain, "resi": [f"{target_resi_start}-{target_resi_end}"]})
view.show()

print(f"\nBlue: target region ({target_chain_residues})")
print(f"Red:  hotspot residues ({hotspot_residues})")
print(f"Gray: non-target region")

---
## **3. バインダーデザインの実行**

In [ ]:
#@title 3.1 デザインパイプラインの設定
#@markdown - **num_samples**: 生成するバインダー候補の数
#@markdown - **search_algorithm**: 探索アルゴリズム（デモでは single-pass 推奨）
#@markdown - **seed**: 乱数シード（再現性のため）
#@markdown - **ncpus**: 使用するCPUコア数
#@markdown
#@markdown ---
num_samples = 5 #@param {type:"integer"}
search_algorithm = "single-pass" #@param ["single-pass", "best-of-n"]
seed = 1101 #@param {type:"integer"}
ncpus = 2 #@param {type:"integer"}

from pathlib import Path

COMPLEXA_DIR = Path("/content/Proteina-Complexa")
SAMPLES_DIR = PROJECT_DIR / "results" / "samples"
LOGS_DIR = PROJECT_DIR / "logs"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

pipeline_yaml = f"""defaults:
  - pipeline/binder/binder_generate@generation
  - _self_

run_name: {project_name}
ckpt_path: {COMPLEXA_DIR / "ckpts"}
ckpt_name: complexa.ckpt
autoencoder_ckpt_path: {COMPLEXA_DIR / "ckpts" / "complexa_ae.ckpt"}
ncpus_: {ncpus}
seed: {seed}
gen_njobs: 1
eval_njobs: 1

hydra:
  run:
    dir: {LOGS_DIR}
"""

pipeline_path = COMPLEXA_DIR / "configs" / "user_pipeline.yaml"
pipeline_path.write_text(pipeline_yaml)
print(f"Pipeline config written: {pipeline_path}")
print(f"Samples: {num_samples}, Algorithm: {search_algorithm}, Seed: {seed}")
print(f"Output: {PROJECT_DIR}")


In [ ]:
#@title 3.2 デザインの実行
#@markdown T4 GPUでは数分から十数分かかります。
#@markdown
#@markdown ---
import os

os.chdir("/content/Proteina-Complexa")

cmd = (
    f"complexa generate configs/user_pipeline.yaml"
    f" ++root_path={SAMPLES_DIR}"
    f" ++generation.task_name={TASK_NAME}"
    f" ++generation.search.algorithm={search_algorithm}"
    f" ++generation.dataloader.batch_size={num_samples}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.source=user_targets"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_filename={target_config['target_filename']}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_path={target_config['target_path']}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_input={target_config['target_input']}"
    f" '++generation.target_dict_cfg.{TASK_NAME}.hotspot_residues={target_config["hotspot_residues"]}'"
    f" '++generation.target_dict_cfg.{TASK_NAME}.binder_length={target_config["binder_length"]}'"
    f" ++generation.target_dict_cfg.{TASK_NAME}.pdb_id={target_config['pdb_id']}"
    f" ++generation.reward_model=null"
)

print(f"Running: {cmd[:100]}...")
print()
run(cmd, "Binder generation")

# --- Results ---
pdb_files = sorted(SAMPLES_DIR.rglob("*.pdb")) if SAMPLES_DIR.exists() else []

if pdb_files:
    print(f"\nGenerated {len(pdb_files)} binder candidates:")
    for p in pdb_files:
        print(f"  {p.relative_to(SAMPLES_DIR)}")
    print(f"\nOutput: {SAMPLES_DIR}")
else:
    print("\nPDB files not found. Check the generation log above for errors.")
    print(f"Searched: {SAMPLES_DIR}")

In [ ]:
#@title 3.3 デザイン結果の3D表示
#@markdown ---
view_index = 0 #@param {type:"integer"}

import py3Dmol
from pathlib import Path

SAMPLES_DIR = PROJECT_DIR / "results" / "samples"
pdb_files = sorted(SAMPLES_DIR.rglob("*.pdb"))

if not pdb_files:
    print("PDBファイルが見つかりません。先に 3.2 を実行してください。")
elif view_index >= len(pdb_files):
    print(f"Index {view_index} is out of range. {len(pdb_files)} files available (0-{len(pdb_files)-1}).")
else:
    pdb_path = pdb_files[view_index]
    print(f"Showing: {pdb_path.name} ({view_index + 1}/{len(pdb_files)})")

    with open(pdb_path) as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_data, "pdb")

    view.setStyle({"chain": "A"}, {"cartoon": {"color": "skyblue"}})
    view.setStyle({"chain": "B"}, {"cartoon": {"color": "lightgreen"}})

    view.zoomTo()
    view.show()

    print(f"\nBlue: target, Green: designed binder")

---
## **4. 構造バリデーション**

設計したバインダーの配列を構造予測モデル（AF2-Multimer / Boltz-1）でリフォールディングし、
設計構造との自己一貫性（self-consistency）を検証します。

In [ ]:
#@title 4.1 構造バリデーション
#@markdown 設計したバインダーをリフォールディングし、自己一貫性（self-consistency）を検証します。
#@markdown
#@markdown ---
#@markdown **注意**: Boltz-1 を使用する場合、MSA計算のためにターゲットチェインのアミノ酸配列が
#@markdown 外部サーバー（api.colabfold.com）に送信されます。
#@markdown ---
use_af2 = True #@param {type:"boolean"}
use_boltz1 = False #@param {type:"boolean"}

import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser, Superimposer
from Bio.PDB.Polypeptide import PPBuilder
import warnings
warnings.filterwarnings("ignore")

os.chdir("/content/Proteina-Complexa")

SAMPLES_DIR = PROJECT_DIR / "results" / "samples"
pdb_files = sorted(SAMPLES_DIR.rglob("*.pdb"))

if not use_af2 and not use_boltz1:
    print("AF2 または Boltz-1 のいずれかを選択してください。")
elif not pdb_files:
    print("PDB ファイルが見つかりません。先に 3.2 を実行してください。")
else:
    parser = PDBParser(QUIET=True)
    ppb = PPBuilder()

    def compute_scrmsd_and_plddt(des_struct, pred_pdb_path, target_chain_ids, binder_chain_id):
        pred_struct = parser.get_structure("pred", str(pred_pdb_path))
        pred_chains = list(pred_struct[0].get_chains())
        pred_binder = pred_chains[-1]

        # pLDDT_binder from B-factors (0-100 -> 0-1)
        bfs = [r["CA"].get_bfactor() for r in pred_binder.get_residues()
               if r.id[0] == " " and "CA" in r]
        plddt_b = float(np.mean(bfs)) / 100.0 if bfs else 0.0

        # scRMSD: align on target CA, measure binder CA RMSD
        des_ca, pred_ca = [], []
        for des_cid, pred_chain in zip(target_chain_ids, pred_chains[:-1]):
            des_map = {r.id[1]: r["CA"] for r in des_struct[0][des_cid].get_residues()
                       if r.id[0] == " " and "CA" in r}
            for r in pred_chain.get_residues():
                if r.id[0] == " " and "CA" in r and r.id[1] in des_map:
                    des_ca.append(des_map[r.id[1]])
                    pred_ca.append(r["CA"])

        scrmsd = float("nan")
        if len(des_ca) >= 3:
            sup = Superimposer()
            sup.set_atoms(des_ca, pred_ca)
            sup.apply(pred_struct[0].get_atoms())
            des_bca = [r["CA"] for r in des_struct[0][binder_chain_id].get_residues()
                       if r.id[0] == " " and "CA" in r]
            pred_bca = [r["CA"] for r in pred_binder.get_residues()
                        if r.id[0] == " " and "CA" in r]
            n = min(len(des_bca), len(pred_bca))
            if n > 0:
                d = np.array([a.coord for a in des_bca[:n]]) - np.array([a.coord for a in pred_bca[:n]])
                scrmsd = float(np.sqrt(np.mean(np.sum(d**2, axis=1))))

        return scrmsd, plddt_b

    # Parse all design structures
    samples = []
    for pdb_path in pdb_files:
        des_struct = parser.get_structure(pdb_path.stem, str(pdb_path))
        chains = list(des_struct[0].get_chains())
        binder_cid = chains[-1].id
        target_cids = [c.id for c in chains[:-1]]
        peps = ppb.build_peptides(des_struct[0][binder_cid])
        binder_seq = "".join(str(pp.get_sequence()) for pp in peps)
        samples.append({
            "pdb_path": pdb_path, "name": pdb_path.stem,
            "des_struct": des_struct, "binder_cid": binder_cid,
            "target_cids": target_cids, "binder_seq": binder_seq,
            "length": len(binder_seq),
        })

    results = {s["name"]: {"name": s["name"], "length": s["length"], "binder_sequence": s["binder_seq"]} for s in samples}

    # ── AF2-Multimer ──
    if use_af2:
        from colabdesign import mk_afdesign_model, clear_mem

        AF2_OUTPUT = PROJECT_DIR / "results" / "af2"
        AF2_OUTPUT.mkdir(parents=True, exist_ok=True)

        print("=" * 70)
        print("  AF2-Multimer Validation")
        print("=" * 70)

        for i, s in enumerate(samples):
            name = s["name"]
            print(f"\n[{i+1}/{len(samples)}] {name}")
            print(f"  Binder: chain {s['binder_cid']} ({s['length']} aa)")

            try:
                clear_mem()
                af_model = mk_afdesign_model(
                    protocol="binder", use_multimer=True, num_recycles=3,
                    data_dir=str(AF2_PARAMS_DIR),
                )
                af_model.prep_inputs(
                    pdb_filename=str(target_pdb),
                    chain=",".join(s["target_cids"]),
                    binder_len=s["length"],
                )
                af_model.predict(seq=s["binder_seq"], models=[0], num_recycles=3)

                pred_pdb = AF2_OUTPUT / f"{name}_af2.pdb"
                af_model.save_pdb(str(pred_pdb))

                log = af_model.aux["log"]
                plddt_b = float(log.get("plddt", 0))
                iptm = float(log.get("i_ptm", 0))
                ipae = float(log.get("i_pae", 0))
                ipsae = float(log.get("min_ipsae", log.get("i_pae", 0)))
                scrmsd, _ = compute_scrmsd_and_plddt(
                    s["des_struct"], pred_pdb, s["target_cids"], s["binder_cid"]
                )

                results[name].update({
                    "af2_pLDDT_binder": plddt_b, "af2_ipTM": iptm,
                    "af2_ipAE": ipae, "af2_ipSAE": ipsae, "af2_scRMSD": scrmsd,
                })
                print(f"  pLDDT_binder={plddt_b:.2f}  ipTM={iptm:.3f}  ipAE={ipae:.2f}  ipSAE={ipsae:.3f}  scRMSD={scrmsd:.2f} Å")

            except Exception as e:
                print(f"  Error: {e}")
                results[name].update({
                    "af2_pLDDT_binder": float("nan"), "af2_ipTM": float("nan"),
                    "af2_ipAE": float("nan"), "af2_ipSAE": float("nan"),
                    "af2_scRMSD": float("nan"),
                })

    # ── Boltz-1 ──
    if use_boltz1:
        BOLTZ_DIR = PROJECT_DIR / "results" / "boltz"
        BOLTZ_INPUT = BOLTZ_DIR / "input"
        BOLTZ_INPUT.mkdir(parents=True, exist_ok=True)

        print("\n" + "=" * 70)
        print("  Boltz-1 Validation")
        print("=" * 70)
        print("  注意: ターゲットチェインの配列を外部サーバー（api.colabfold.com）に送信します。")

        # Prepare YAML: target chains get MSA (server), binder gets msa: empty
        for s in samples:
            yaml_lines = ["version: 1", "sequences:"]
            for chain in s["des_struct"][0]:
                peps = ppb.build_peptides(chain)
                if not peps:
                    continue
                seq = "".join(str(pp.get_sequence()) for pp in peps)
                yaml_lines.append("  - protein:")
                yaml_lines.append(f"      id: {chain.id}")
                yaml_lines.append(f"      sequence: {seq}")
                if chain.id == s["binder_cid"]:
                    yaml_lines.append("      msa: empty")
            (BOLTZ_INPUT / f"{s['name']}.yaml").write_text("\n".join(yaml_lines) + "\n")

        print(f"\nValidating {len(samples)} samples...")
        cmd = (
            f"boltz predict {BOLTZ_INPUT}"
            f" --model boltz1"
            f" --use_msa_server"
            f" --out_dir {BOLTZ_DIR}"
            f" --recycling_steps 3"
            f" --diffusion_samples 1"
            f" --output_format pdb"
            f" --override"
        )
        run(cmd, "Boltz-1 structure prediction")

        pred_dir = BOLTZ_DIR / "predictions"
        for s in samples:
            name = s["name"]
            sample_dir = pred_dir / name
            conf_files = sorted(sample_dir.glob("confidence_*.json")) if sample_dir.exists() else []
            pred_pdbs = sorted(sample_dir.glob("*.pdb")) if sample_dir.exists() else []

            if conf_files and pred_pdbs:
                with open(conf_files[0]) as f:
                    conf = json.load(f)
                iptm = float(conf.get("iptm", 0))
                scrmsd, plddt_b = compute_scrmsd_and_plddt(
                    s["des_struct"], pred_pdbs[0], s["target_cids"], s["binder_cid"]
                )
                results[name].update({
                    "boltz1_pLDDT_binder": plddt_b, "boltz1_ipTM": iptm,
                    "boltz1_scRMSD": scrmsd,
                })
            else:
                results[name].update({
                    "boltz1_pLDDT_binder": float("nan"), "boltz1_ipTM": float("nan"),
                    "boltz1_scRMSD": float("nan"),
                })

    # ── Save CSV ──
    df = pd.DataFrame(list(results.values()))
    csv_path = PROJECT_DIR / "results" / "validation_scores.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nCSV saved: {csv_path}")

    # ── Summary tables ──
    if use_af2:
        print("\n" + "=" * 78)
        print("  [AF2-Multimer]")
        print("=" * 78)
        print(f"  {'Sample':<25} {'len':>4} {'pLDDT_b':>7} {'ipTM':>6} {'ipAE':>6} {'ipSAE':>6} {'scRMSD':>8}")
        print("  " + "─" * 72)
        for r in results.values():
            rmsd = r.get("af2_scRMSD", float("nan"))
            rmsd_s = f"{rmsd:.2f} Å" if not np.isnan(rmsd) else "  N/A"
            print(f"  {r['name']:<25} {r['length']:>4}"
                  f" {r.get('af2_pLDDT_binder', float('nan')):>7.2f}"
                  f" {r.get('af2_ipTM', float('nan')):>6.3f}"
                  f" {r.get('af2_ipAE', float('nan')):>6.2f}"
                  f" {r.get('af2_ipSAE', float('nan')):>6.3f}"
                  f" {rmsd_s:>8}")

    if use_boltz1:
        print("\n" + "=" * 65)
        print("  [Boltz-1]")
        print("=" * 65)
        print(f"  {'Sample':<25} {'len':>4} {'pLDDT_b':>7} {'ipTM':>6} {'scRMSD':>8}")
        print("  " + "─" * 56)
        for r in results.values():
            rmsd = r.get("boltz1_scRMSD", float("nan"))
            rmsd_s = f"{rmsd:.2f} Å" if not np.isnan(rmsd) else "  N/A"
            print(f"  {r['name']:<25} {r['length']:>4}"
                  f" {r.get('boltz1_pLDDT_binder', float('nan')):>7.2f}"
                  f" {r.get('boltz1_ipTM', float('nan')):>6.3f}"
                  f" {rmsd_s:>8}")

    print(f"\n  閾値の調整は 4.3 フィルタリング で行います。")

In [ ]:
#@title 4.2 スコアの解析
#@markdown バリデーション結果を可視化します。
#@markdown ---

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

csv_path = PROJECT_DIR / "results" / "validation_scores.csv"

if not csv_path.exists():
    print("validation_scores.csv が見つかりません。先に 4.1 を実行してください。")
else:
    df = pd.read_csv(csv_path)
    has_af2 = "af2_ipTM" in df.columns
    has_boltz1 = "boltz1_ipTM" in df.columns

    methods = []
    if has_af2:
        methods.append(("AF2", "af2"))
    if has_boltz1:
        methods.append(("Boltz-1", "boltz1"))

    # ── 1. Score table ──
    show_cols = ["name", "length"]
    for _, pfx in methods:
        if pfx == "af2":
            show_cols += [f"{pfx}_pLDDT_binder", f"{pfx}_ipTM", f"{pfx}_ipAE", f"{pfx}_ipSAE", f"{pfx}_scRMSD"]
        else:
            show_cols += [f"{pfx}_pLDDT_binder", f"{pfx}_ipTM", f"{pfx}_scRMSD"]
    show_cols = [c for c in show_cols if c in df.columns]
    display(df[show_cols].style.format(precision=3, na_rep="N/A"))

    # ── 2. Scatter: ipSAE vs scRMSD (AF2) / ipTM vs scRMSD (Boltz-1) ──
    n_m = len(methods)
    fig, axes = plt.subplots(1, n_m, figsize=(5.5 * n_m, 4.5), squeeze=False)

    for idx, (label, pfx) in enumerate(methods):
        ax = axes[0, idx]
        y = df[f"{pfx}_scRMSD"]

        if pfx == "af2" and f"{pfx}_ipSAE" in df.columns:
            x = df[f"{pfx}_ipSAE"]
            x_label = "ipSAE"
        else:
            x = df[f"{pfx}_ipTM"]
            x_label = "ipTM"

        ax.scatter(x, y, s=70, c="steelblue", edgecolors="black", linewidth=0.5, zorder=3)

        for i, nm in enumerate(df["name"]):
            short = nm[:18] + "..." if len(nm) > 18 else nm
            ax.annotate(short, (x.iloc[i], y.iloc[i]),
                        fontsize=7, textcoords="offset points", xytext=(5, 5))

        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel("scRMSD (Å)", fontsize=11)
        ax.set_title(f"{label}: {x_label} vs scRMSD", fontsize=12, fontweight="bold")
        if y.notna().any():
            ax.set_ylim(bottom=-0.2)

    plt.tight_layout()
    plt.show()

    # ── 3. Per-sample metric bars ──
    for label, pfx in methods:
        metric_cols = [("pLDDT_binder", "#3498db"), ("ipTM", "#2ecc71")]

        fig, ax = plt.subplots(figsize=(max(6, len(df) * 1.2), 3.5))
        x_pos = np.arange(len(df))
        width = 0.35

        for j, (m, color) in enumerate(metric_cols):
            col = f"{pfx}_{m}"
            if col in df.columns:
                offset = (j - len(metric_cols) / 2 + 0.5) * width
                ax.bar(x_pos + offset, df[col], width, label=m, color=color, alpha=0.85, edgecolor="white")

        ax.set_xticks(x_pos)
        ax.set_xticklabels([n[:20] for n in df["name"]], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Score", fontsize=11)
        ax.set_title(f"{label}: Per-sample Metrics", fontsize=12, fontweight="bold")
        ax.legend(loc="upper right", fontsize=9)
        ax.set_ylim(0, 1.08)
        plt.tight_layout()
        plt.show()

    # ── 4. AF2 vs Boltz-1 comparison ──
    if has_af2 and has_boltz1:
        fig, axes = plt.subplots(1, 2, figsize=(9, 4))

        ax = axes[0]
        ax.scatter(df["af2_ipTM"], df["boltz1_ipTM"], s=60, c="steelblue", edgecolors="black", lw=0.5)
        ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4)
        for i, nm in enumerate(df["name"]):
            ax.annotate(nm[:15], (df["af2_ipTM"].iloc[i], df["boltz1_ipTM"].iloc[i]),
                        fontsize=7, textcoords="offset points", xytext=(4, 4))
        ax.set_xlabel("AF2 ipTM", fontsize=11)
        ax.set_ylabel("Boltz-1 ipTM", fontsize=11)
        ax.set_title("ipTM: AF2 vs Boltz-1", fontsize=12, fontweight="bold")
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)

        ax = axes[1]
        mask = df["af2_scRMSD"].notna() & df["boltz1_scRMSD"].notna()
        if mask.any():
            ax.scatter(df.loc[mask, "af2_scRMSD"], df.loc[mask, "boltz1_scRMSD"],
                       s=60, c="coral", edgecolors="black", lw=0.5)
            lim = max(df.loc[mask, "af2_scRMSD"].max(), df.loc[mask, "boltz1_scRMSD"].max()) * 1.2
            ax.plot([0, lim], [0, lim], "k--", lw=0.8, alpha=0.4)
            for i in df[mask].index:
                ax.annotate(df["name"].iloc[i][:15],
                            (df["af2_scRMSD"].iloc[i], df["boltz1_scRMSD"].iloc[i]),
                            fontsize=7, textcoords="offset points", xytext=(4, 4))
        ax.set_xlabel("AF2 scRMSD (Å)", fontsize=11)
        ax.set_ylabel("Boltz-1 scRMSD (Å)", fontsize=11)
        ax.set_title("scRMSD: AF2 vs Boltz-1", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()

In [ ]:
#@title 4.3 フィルタリング
#@markdown 4.2 の結果を参考に閾値を調整し、候補をフィルタリングします。
#@markdown
#@markdown ---
#@markdown ### AF2 閾値
#@markdown - **ipSAE**: interface predicted Symmetrized Aligned Error（低いほど良い）
af2_ipSAE_max = 0.5 #@param {type:"number"}
af2_scRMSD_max = 2.0 #@param {type:"number"}
af2_pLDDT_min = 0.7 #@param {type:"number"}
#@markdown ### Boltz-1 閾値
#@markdown - **ipTM**: interface predicted TM-score（高いほど良い）
boltz1_ipTM_min = 0.5 #@param {type:"number"}
boltz1_scRMSD_max = 2.0 #@param {type:"number"}
boltz1_pLDDT_min = 0.7 #@param {type:"number"}

import pandas as pd
import numpy as np
from IPython.display import display

csv_path = PROJECT_DIR / "results" / "validation_scores.csv"

if not csv_path.exists():
    print("validation_scores.csv が見つかりません。先に 4.1 を実行してください。")
else:
    df = pd.read_csv(csv_path)
    has_af2 = "af2_ipTM" in df.columns
    has_boltz1 = "boltz1_ipTM" in df.columns

    # ── Pass / Fail 判定 ──
    if has_af2:
        df["af2_pass"] = (
            (df["af2_ipSAE"] < af2_ipSAE_max)
            & (df["af2_scRMSD"] < af2_scRMSD_max)
            & (df["af2_pLDDT_binder"] > af2_pLDDT_min)
        )
    if has_boltz1:
        df["boltz1_pass"] = (
            (df["boltz1_ipTM"] > boltz1_ipTM_min)
            & (df["boltz1_scRMSD"] < boltz1_scRMSD_max)
            & (df["boltz1_pLDDT_binder"] > boltz1_pLDDT_min)
        )

    # ── Styled table ──
    show_cols = ["name", "length"]
    if has_af2:
        show_cols += ["af2_pLDDT_binder", "af2_ipSAE", "af2_scRMSD", "af2_pass"]
    if has_boltz1:
        show_cols += ["boltz1_pLDDT_binder", "boltz1_ipTM", "boltz1_scRMSD", "boltz1_pass"]
    show_cols = [c for c in show_cols if c in df.columns]

    def color_pass(val):
        if val is True or val == True:
            return "background-color: #d4edda"
        if val is False or val == False:
            return "background-color: #f8d7da"
        return ""

    pass_cols = [c for c in show_cols if c.endswith("_pass")]
    styled = df[show_cols].style.map(color_pass, subset=pass_cols).format(precision=3, na_rep="N/A")
    display(styled)

    # ── Summary ──
    print()
    if has_af2:
        n = df["af2_pass"].sum()
        print(f"AF2:     {n}/{len(df)} passed  (ipSAE < {af2_ipSAE_max}, scRMSD < {af2_scRMSD_max} Å, pLDDT > {af2_pLDDT_min})")
    if has_boltz1:
        n = df["boltz1_pass"].sum()
        print(f"Boltz-1: {n}/{len(df)} passed  (ipTM > {boltz1_ipTM_min}, scRMSD < {boltz1_scRMSD_max} Å, pLDDT > {boltz1_pLDDT_min})")

    # ── Passed candidates ──
    pass_any = pd.Series(False, index=df.index)
    if has_af2:
        pass_any |= df["af2_pass"]
    if has_boltz1:
        pass_any |= df["boltz1_pass"]

    if pass_any.any():
        print(f"\nPassed candidates:")
        for _, row in df[pass_any].iterrows():
            tags = []
            if has_af2 and row.get("af2_pass"):
                tags.append("AF2")
            if has_boltz1 and row.get("boltz1_pass"):
                tags.append("Boltz-1")
            print(f"  {row['name']}  ({', '.join(tags)})")
    else:
        print("\nPass した候補がありません。閾値を緩和して再実行してください。")

In [ ]:
#@title 4.4 リフォールディング結果の3D表示
#@markdown ---
view_method = "AF2" #@param ["AF2", "Boltz-1"]
view_index = 0 #@param {type:"integer"}

import py3Dmol
from pathlib import Path

SAMPLES_DIR = PROJECT_DIR / "results" / "samples"
pdb_files = sorted(SAMPLES_DIR.rglob("*.pdb"))

if not pdb_files:
    print("PDBファイルが見つかりません。")
elif view_index >= len(pdb_files):
    print(f"Index {view_index} is out of range. {len(pdb_files)} files available (0-{len(pdb_files)-1}).")
else:
    name = pdb_files[view_index].stem

    if view_method == "AF2":
        pred_pdb = PROJECT_DIR / "results" / "af2" / f"{name}_af2.pdb"
    else:
        pred_dir = PROJECT_DIR / "results" / "boltz" / "predictions" / name
        pred_pdbs = sorted(pred_dir.glob("*.pdb")) if pred_dir.exists() else []
        pred_pdb = pred_pdbs[0] if pred_pdbs else None

    if pred_pdb is None or not pred_pdb.exists():
        print(f"{view_method} predicted structure not found for {name}")
        print("先に 4.1 を実行してください。")
    else:
        print(f"{view_method} prediction: {name}")

        with open(pred_pdb) as f:
            pred_data = f.read()

        view = py3Dmol.view(width=800, height=500)
        view.addModel(pred_data, "pdb")
        view.setStyle({"chain": "A"}, {"cartoon": {"color": "skyblue"}})
        view.setStyle({"chain": "B"}, {"cartoon": {"color": "lightgreen"}})
        view.zoomTo()
        view.show()

        print(f"\nBlue: target, Green: predicted binder")